In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torchvision
from torchvision.datasets import mnist

transform = torchvision.transforms.Compose([
    torchvision.transforms.Grayscale(num_output_channels=1),
    torchvision.transforms.ToTensor(),
    # Removed normalization to keep data in [0, 1] range for consistency with Noising function and loss
    # torchvision.transforms.Normalize((0.5,), (0.5,))
])

# Load the full MNIST training data
full_train_data = mnist.MNIST(root="./data", train=True, transform=transform, download=True)

# Filter the dataset to include only images with label 7
seven_indices = [i for i, (image, label) in enumerate(full_train_data) if label == 7]
train_data_seven = torch.utils.data.Subset(full_train_data, seven_indices)

# Use the filtered dataset for training
train_data = train_data_seven

In [ ]:
import numpy as np
import torch
from itertools import product
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt

# Logistic map
def logistic_forward(x: torch.Tensor, num_steps: int, theta: float = 4.0, device: str = "cpu"):
    x = x.to(device)
    for _ in range(num_steps):
        x = theta * x * (1 - x)
    return x

def logistic_inverse(x: torch.Tensor, num_steps: int, theta: float = 4.0, device: str = "cuda"):
    """
    Logistic inverse using a binary tree approach.
    Expands real-valued branches only, stops when:
      - discriminant < 0 (complex)
      - value already appeared (repeating leaf)
    """
    x = x.to(device)
    active_branches = [x]  # list of current nodes
    seen = set()            # to store hashes of seen branches

    for step in range(num_steps):
        new_branches = []

        for branch in active_branches:
            disc = 1 - 4 * branch / theta
            real_mask = disc >= 0
            if not torch.any(real_mask):
                continue  # skip complex branch entirely

            disc = torch.clamp(disc, min=0.0)
            root = torch.sqrt(disc)

            # Two possible inverse branches
            x1 = (1 + root) / 2
            x2 = (1 - root) / 2

            for child in [x1, x2]:
                # Make hashable key (approximate via rounding)
                key = torch.round(child * 1e6).detach().cpu().numpy().tobytes()
                if key in seen:
                    continue  # skip repeated branch
                seen.add(key)
                new_branches.append(child)

        if not new_branches:
            break  # no valid new branches, stop
        active_branches = new_branches  # move to next level

    # Stack final valid leaves
    solns = torch.stack(active_branches, dim=0) if active_branches else torch.empty(0, *x.shape, device=device)
    return solns.to(device)

In [ ]:
import torch
import torch.nn as nn

class SLP(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(SLP, self).__init__()
        # Linear layer with no bias
        self.linear = nn.Linear(input_dim, output_dim, bias=False)

    def forward(self, x):
        # No activation function applied
        return self.linear(x)

# Example usage:
# input_dim = 784 # e.g., for flattened MNIST image
# output_dim = 10 # e.g., for 10 classes
# model = SingleLayerPerceptron(input_dim, output_dim)
# print(model)

In [ ]:
from torch.utils.data import DataLoader
import torch.optim as optim
import torch
import random
import torch.nn as nn
import numpy as np

# Logistic map
def logistic_forward(x: torch.Tensor, num_steps: int, theta: float = 4.0, device: str = "cpu"):
    x = x.to(device)
    for _ in range(num_steps):
        x = theta * x * (1 - x)
    return x

def logistic_inverse(x: torch.Tensor, num_steps: int, theta: float = 4.0, device: str = "cuda"):
    """
    Logistic inverse using a binary tree approach.
    Expands real-valued branches only, stops when:
      - discriminant < 0 (complex)
      - value already appeared (repeating leaf)
    """
    x = x.to(device)
    active_branches = [x]  # list of current nodes
    seen = set()            # to store hashes of seen branches

    for step in range(num_steps):
        new_branches = []

        for branch in active_branches:
            disc = 1 - 4 * branch / theta
            real_mask = disc >= 0
            if not torch.any(real_mask):
                continue  # skip complex branch entirely

            disc = torch.clamp(disc, min=0.0)
            root = torch.sqrt(disc)

            # Two possible inverse branches
            x1 = (1 + root) / 2
            x2 = (1 - root) / 2

            for child in [x1, x2]:
                # Make hashable key (approximate via rounding)
                key = torch.round(child * 1e6).detach().cpu().numpy().tobytes()
                if key in seen:
                    continue  # skip repeated branch
                seen.add(key)
                new_branches.append(child)

        if not new_branches:
            break  # no valid new branches, stop
        active_branches = new_branches  # move to next level

    # Stack final valid leaves
    solns = torch.stack(active_branches, dim=0) if active_branches else torch.empty(0, *x.shape, device=device)
    return solns.to(device)


class SLP(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(SLP, self).__init__()
        # Linear layer with no bias
        self.linear = nn.Linear(input_dim, output_dim, bias=False)

    def forward(self, x):
        # No activation function applied
        return self.linear(x)


def train(train_data, num_epochs, noising_steps, model, batch_size=16, device="cpu"):
    model.train()
    criterion = nn.L1Loss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    train_data = DataLoader(train_data, batch_size=batch_size, shuffle=True)

    # Define the regularization strength (lambda)
    lambda_l1 = 0.001

    for epoch in range(num_epochs):
        total_loss = 0.0

        for batch_idx, (images, labels) in enumerate(train_data):

            images = images.to(device)
            labels = labels.to(device)

            noisy_images = logistic_forward(images.view(-1, 784), num_steps=noising_steps, device=device)
            solution_space = logistic_inverse(noisy_images, num_steps=noising_steps, device=device)

            # Remove the extra dimension from solution_space
            # print(solution_space.shape)
            solution_space = solution_space.squeeze(1)

            images_out = model(solution_space)

            loss = criterion(images_out, images.view(-1, 784))

            # Add L1 regularization
            l1_reg = torch.tensor(0.).to(device)
            for param in model.parameters():
                l1_reg += torch.norm(param, 1)

            loss = loss + lambda_l1 * l1_reg


            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * images.size(0)

        avg_loss = total_loss / len(train_data.dataset)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.8f}")


    torch.save(model.state_dict(), 'diffusion_deepmodel_model.pth')
    print("model weights saved to diffusion_deepmodel_model.pth")


# Pass the desired noising_steps to both the model definition and the training function
noising_steps = 5
model = SLP(input_dim=784, output_dim=784)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

train(train_data=train_data, num_epochs=20, noising_steps=noising_steps, model=model, device=device)

Epoch [1/20], Loss: 1.20669029
Epoch [2/20], Loss: 1.04740560
Epoch [3/20], Loss: 1.04169123
Epoch [4/20], Loss: 1.03785076
Epoch [5/20], Loss: 1.03840010
Epoch [6/20], Loss: 1.03636392
Epoch [7/20], Loss: 1.03789741
Epoch [8/20], Loss: 1.03912573
Epoch [9/20], Loss: 1.03322703
Epoch [10/20], Loss: 1.03252941
Epoch [11/20], Loss: 1.03191358
Epoch [12/20], Loss: 1.03295328
Epoch [13/20], Loss: 1.03141815
Epoch [14/20], Loss: 1.03672519
Epoch [15/20], Loss: 1.03669485
Epoch [16/20], Loss: 1.03664465
Epoch [17/20], Loss: 1.03555447
Epoch [18/20], Loss: 1.03587757
Epoch [19/20], Loss: 1.03504027
Epoch [20/20], Loss: 1.03369791
model weights saved to diffusion_deepmodel_model.pth


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Generate random image from beta(0.5,0.5)
random_image = torch.random.beta(0.5, 0.5)
random_image = random_image.to(device)

solution_space = logistic_inverse(random_image, num_steps=1)
model.eval()
with torch.no_grad():
    reconstructed_image = model(solution_space)

# Reshape and display images using matplotlib
random_image_np = random_image.squeeze().cpu().numpy()
reconstructed_image_np = reconstructed_image.squeeze().cpu().numpy()

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(random_image_np, cmap='gray')
plt.title("Random Image")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(reconstructed_image_np, cmap='gray')
plt.title("Reconstructed Image")
plt.axis('off')

plt.show()

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn # Import nn module
from torch.distributions import Beta # Import Beta distribution

# Define the SLP model class (assuming it's the same as used for training)
class SLP(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(SLP, self).__init__()
        self.linear = nn.Linear(input_dim, output_dim, bias=False)

    def forward(self, x):
        return self.linear(x)

# Define the logistic_inverse function (assuming it's the same as used for training)
def logistic_inverse(x: torch.Tensor, num_steps: int, theta: float = 4.0, device: str = "cuda"):
    """
    Logistic inverse using a binary tree approach.
    Expands real-valued branches only, stops when:
      - discriminant < 0 (complex)
      - value already appeared (repeating leaf)
    """
    x = x.to(device)
    active_branches = [x]  # list of current nodes
    seen = set()            # to store hashes of seen branches

    for step in range(num_steps):
        new_branches = []

        for branch in active_branches:
            disc = 1 - 4 * branch / theta
            real_mask = disc >= 0
            if not torch.any(real_mask):
                continue  # skip complex branch entirely

            disc = torch.clamp(disc, min=0.0)
            root = torch.sqrt(disc)

            # Two possible inverse branches
            x1 = (1 + root) / 2
            x2 = (1 - root) / 2

            for child in [x1, x2]:
                # Make hashable key (approximate via rounding)
                key = torch.round(child * 1e6).detach().cpu().numpy().tobytes()
                if key in seen:
                    continue  # skip repeated branch
                seen.add(key)
                new_branches.append(child)

        if not new_branches:
            break  # no valid new branches, stop
        active_branches = new_branches  # move to next level

    # Stack final valid leaves
    solns = torch.stack(active_branches, dim=0) if active_branches else torch.empty(0, *x.shape, device=device)
    return solns.to(device)


# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define model dimensions to match the loaded weights (assuming noising_steps was 0 during training)
# To load the saved model, input_dim must be 784. This corresponds to noising_steps = 0.
model_noising_steps = 5
input_dim = 784 * (2**model_noising_steps) # Should be 784
output_dim = 784

# Create model instance
model = SLP(input_dim=input_dim, output_dim=output_dim)
model.to(device)

# Load the saved model weights
model.load_state_dict(torch.load('diffusion_deepmodel_model.pth'))
model.eval() # Set model to evaluation mode

# Set the desired noising steps for image generation
# IMPORTANT: This model was trained with noising_steps = 0.
# Generating with a different number of steps will likely not produce meaningful results.
generation_noising_steps = 0 # Set to 0 to match the trained model


# Generate 20 random images from beta(0.5, 0.5)
num_images = 20
# Fix: Use torch.distributions.Beta to generate from beta distribution
m = Beta(torch.tensor([0.5]), torch.tensor([0.5]))
random_images = m.sample((num_images, 784)).to(device)


# Process each random image through the logistic inverse and the model
reconstructed_images = []
for img in random_images:
    # Apply logistic inverse using the generation_noising_steps
    solution_space = logistic_inverse(img.view(1, -1), num_steps=generation_noising_steps, device=device)

    # Reshape solution_space to (1, input_dim) to match the model's expected input
    # With generation_noising_steps=0, solution_space shape should be [1, 1, 784].
    # Permuting and reshaping will result in [1, 784] which matches the model's input_dim=784.
    solution_space = solution_space.permute(1, 0, 2).contiguous().view(1, -1).to(device)


    # Reconstruct image using the model
    with torch.no_grad():
        reconstructed_image = model(solution_space)
    reconstructed_images.append(reconstructed_image.squeeze().cpu().numpy())

random_images_np = random_images.squeeze().cpu().numpy()

# Display the original random images and the reconstructed images
plt.figure(figsize=(15, 10))
for i in range(num_images):
    plt.subplot(4, 10, i + 1)
    plt.imshow(random_images_np[i].reshape(28, 28), cmap='gray')
    plt.title("Random")
    plt.axis('off')

    plt.subplot(4, 10, i + 11)
    plt.imshow(reconstructed_images[i].reshape(28, 28), cmap='gray')
    plt.title("Reconstructed")
    plt.axis('off')

plt.tight_layout()
plt.show()

RuntimeError: Error(s) in loading state_dict for SLP:
	size mismatch for linear.weight: copying a param with shape torch.Size([784, 784]) from checkpoint, the shape in current model is torch.Size([784, 25088]).